# Module B: Cancellation Risk (Classification), Part 2

**Goal:** finish Module B. Add the two strongest model families (Gradient Boosting, MLP), tune the top two models, choose a decision threshold from a business cost model, evaluate **once** on the held-out test set, explain the model with SHAP, and save `cancel_model.pkl` for the Streamlit app.

**Starting point (Part 1):** the best model was Random Forest with ROC-AUC 0.905 (5-fold CV). The leaky columns `assigned_room_type` and `booking_changes` stay dropped. The split, seed and CV folds are identical to Part 1, so the leaderboards can be compared directly.

| Section | Content |
|---|---|
| 0 | Setup: same split, helpers, baseline, Part 1 leaderboard |
| 1 | Gradient Boosting |
| 2 | MLP neural network |
| 3 | Hyperparameter tuning (top 2 models: GB and RF) |
| 4 | ROC and Precision-Recall curves |
| 5 | Choosing the threshold by business cost (+ calibration check) |
| 6 | Final test-set evaluation (run once) |
| 7 | SHAP explanations |
| 8 | Save the model and the final leaderboard |
| 9 | Summary, decision log and cheat sheet rows |

## 0. Setup

In [ ]:
# Imports: all libraries for this notebook in one place
import sys
sys.path.insert(0, "..")          # project root, so `src` can be found

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import sklearn
from scipy.stats import loguniform, randint

from sklearn.model_selection import (StratifiedKFold, cross_validate, cross_val_predict,
                                     train_test_split, RandomizedSearchCV)
from sklearn.metrics import (make_scorer, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, brier_score_loss,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from src._01_code.features import build_preprocessor

In [ ]:
# Global settings: same seed, split size and targets as Part 1, so every model sees identical data
RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGETS = ["is_canceled", "adr"]

df = pd.read_csv("../data/clean/hotel_clean.csv")

In [ ]:
# Stratified 80/20 split (same as Part 1): keeps the 27.5% cancel rate in both sets; the test set is used ONCE, in Section 6
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["is_canceled"], random_state=RANDOM_STATE
)

# Columns only known AFTER booking (found in Part 1's leakage check): never use as features
LEAKY = ["assigned_room_type", "booking_changes"]

# Features exclude both targets AND the leaky columns
X_train = train_df.drop(columns=TARGETS + LEAKY)
X_test = test_df.drop(columns=TARGETS + LEAKY)
y_train, y_test = train_df["is_canceled"], test_df["is_canceled"]

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

# Safety checks: fail loudly if a leaky or target column slipped in, or train/test columns differ
assert not set(LEAKY + TARGETS) & set(X_train.columns)
assert list(X_train.columns) == list(X_test.columns)
print(f"train {X_train.shape}, test {X_test.shape}, cancel rate {y_train.mean():.3f}")

In [ ]:
# Helpers: make_pipe (preprocessing + model), cv_score (5-fold CV = one leaderboard row), plot_cm (CV confusion matrix)

def make_pipe(model, num=None, cat=None):
    num = num_cols if num is None else num
    cat = cat_cols if cat is None else cat
    return Pipeline([
        ("prep", build_preprocessor(num, cat)),   # scaler/encoder fitted inside each CV fold: no leakage
        ("model", model),
    ])

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# zero_division=0: the baseline never predicts 'cancels', so precision/F1 are 0 instead of a warning flood
SCORING = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
}

def cv_score(name, pipe, X=None, y=None, n_jobs=-1):
    X = X_train if X is None else X
    y = y_train if y is None else y
    cv = cross_validate(pipe, X, y, cv=CV, scoring=SCORING,
                        n_jobs=n_jobs, return_train_score=True)
    row = {"model": name}
    for m in SCORING:
        row[m] = cv[f"test_{m}"].mean()
    row["roc_auc_std"] = cv["test_roc_auc"].std()   # fold-to-fold noise: is a gap between two models real?
    row["train_f1"] = cv["train_f1"].mean()         # compare with CV F1 to spot overfitting
    return row

def plot_cm(name, pipe, X=None, y=None):
    X = X_train if X is None else X
    y = y_train if y is None else y
    y_pred = cross_val_predict(pipe, X, y, cv=CV, n_jobs=-1)
    ConfusionMatrixDisplay.from_predictions(
        y, y_pred, display_labels=["stays", "cancels"],
        normalize="true",             # each row is a share of the true class (bottom-left = missed cancellations)
    )
    plt.title(name)
    plt.show()

In [ ]:
# Majority-class baseline: always predicts 'stays'; every model must beat it on F1 and ROC-AUC
results = [cv_score("Majority baseline", make_pipe(DummyClassifier(strategy="most_frequent")))]
pd.DataFrame(results)

In [ ]:
# Part 1 leaderboard (Day 3), loaded for comparison; the new rows are added to it in Section 8
day3 = pd.read_csv("../reports/day3_leaderboard.csv")
day3[["model", "roc_auc", "f1", "precision", "recall"]].round(3)

### Insight: setup

- **Same data as Part 1.** The seed (42), the stratified 80/20 split and the 5 CV folds are identical, so every Part 2 row can be compared directly with the Part 1 leaderboard.
- **Leaky columns removed before anything else.** `assigned_room_type` and `booking_changes` are dropped in the split cell, and an `assert` stops the notebook if they come back. In an earlier version they slipped into the Gradient Boosting run and inflated ROC-AUC from 0.910 to 0.918.
- **Baseline to beat:** the majority-class model has accuracy 0.725, F1 0 and ROC-AUC 0.50. Accuracy alone would make a useless model look decent, which is why models are compared on ROC-AUC and F1.
- **Preprocessing lives inside the pipeline**, so the scaler and encoder are fitted only on the training part of each fold, with no leakage from the validation rows.

## 1. Gradient Boosting

**Intuition:** a chain of small trees. Each new tree looks only at the bookings the current model still gets wrong and corrects them a little: `F_new(x) = F_old(x) + learning_rate × tree(x)`.

| | Random Forest (bagging) | Gradient Boosting |
|---|---|---|
| Trees built | In parallel, independently | One after another, each fixing the last |
| Tree depth | Deep (low bias, high variance) | Shallow (high bias, low variance) |
| What it fixes | Variance, by averaging | Bias, by adding corrections |
| Key hyperparameters | `n_estimators`, `max_features`, `min_samples_leaf` | `learning_rate`, `max_iter`, `max_leaf_nodes` |
| Needs scaling? | No | No |

`HistGradientBoostingClassifier` bins each feature into at most 255 values, so it trains fast on 69k rows.

In [ ]:
# Gradient Boosting with near-default settings; random_state fixes the internal early-stopping split so results repeat
gradient_boost = make_pipe(HistGradientBoostingClassifier(
    learning_rate=0.1, max_iter=200, random_state=RANDOM_STATE))
results.append(cv_score("Gradient boosting", gradient_boost))
plot_cm("Gradient boosting", gradient_boost)
pd.DataFrame(results)

In [ ]:
# How many trees did it keep? Above 10k rows early stopping switches on by itself (10% of rows held out)
gradient_boost.fit(X_train, y_train)
print("trees used:", gradient_boost.named_steps["model"].n_iter_, "of max_iter=200")

### Insight: Gradient Boosting

- **New leader.** ROC-AUC 0.910 vs 0.905 for the Part 1 Random Forest, and F1 0.698 vs 0.666. Recall at the 0.5 threshold rose from 0.59 to 0.66, so it catches more cancellations without losing much precision (0.74).
- **Less overfitting than the forest.** The gap between train and CV F1 is 0.036, against 0.072 for the forest. Shallow trees that make small corrections overfit less than deep, averaged trees.
- **Leakage check repeated.** A first run with the leaky columns still in gave ROC-AUC 0.918. Removing them cost about 0.008, the same size of drop as for Logistic Regression in Part 1 (0.863 → 0.855). The leak is small, and the honest number is 0.910.
- **It hadn't finished learning.** Early stopping never fired: all 200 trees were used, so the validation score was still improving. That's a signal to allow more trees and to tune the learning rate (Section 3).

## 2. MLP neural network

**Intuition:** Logistic Regression scores a booking with one weighted sum. An MLP stacks layers of these units with a non-linearity (ReLU) in between, so the early layers can build combinations such as "long lead time AND non-refundable deposit" and the last layer scores them. It learns by gradient descent, so it **needs scaled inputs**. The pipeline already scales the numeric columns.

In [ ]:
# MLP: two hidden layers (64, 32); early_stopping holds out 10% and stops when the validation score stops improving
mlp = make_pipe(MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True,
                              max_iter=200, random_state=RANDOM_STATE))
results.append(cv_score("MLP classifier", mlp))
plot_cm("MLP classifier", mlp)
pd.DataFrame(results)

### Insight: MLP

- **Competitive, but not the winner.** ROC-AUC 0.901 and F1 0.685, just behind Gradient Boosting (0.910) and close to the Part 1 Random Forest (0.905). Its confusion matrix is almost identical to the GB one.
- **Why trees still win here.** The data is tabular with many one-hot columns and threshold-like effects (deposit type, parking, previous cancellations). Trees model these splits directly, while the MLP has to approximate them with smooth functions. On tabular data, tree ensembles usually match or beat neural nets, and they are faster to train and easier to explain.
- **Why scaling matters for the MLP but not for trees.** The MLP's first layer computes `w1·lead_time + w2·requests + …`. Unscaled, `lead_time` (0–700) dominates that sum and gets about 100× larger gradient updates than `total_of_special_requests` (0–5), so training is unstable and the small features are ignored. A tree split like `lead_time > 90` depends only on the ordering of values, so rescaling a feature produces the same tree.
- **Decision:** the MLP isn't carried forward. Tuning budget goes to the top two models, GB and RF.

## 3. Hyperparameter tuning (top 2 models)

**Why random search:** with 3 hyperparameters, a 4×4×4 grid needs 64 runs but tries only 4 values of each. 10–20 random settings try 10–20 different values of each, which explores the one or two hyperparameters that really matter (usually `learning_rate`) far better. The risk of too few trials is luck: the "best" setting may only be the best of a poor draw, so read the top 5 rows and not just `best_params_`.

**Laptop-friendly setup:** the full search (20 settings × 5 folds on 69k rows, with `n_jobs=-1` on top of a model that already uses every core) overheated the laptop. So the search runs on a **30k-row stratified sample with 3 folds and one fit at a time**. The search only has to *rank* settings. The winner is then re-scored with the usual 5-fold CV on the **full** training set, so every leaderboard row is still comparable.

In [ ]:
# Search data: a stratified 30k-row sample of the TRAINING set and 3 folds (the test set is never touched)
X_srch, _, y_srch, _ = train_test_split(
    X_train, y_train, train_size=30_000, stratify=y_train, random_state=RANDOM_STATE)

CV_SEARCH = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

### 3.1 Gradient Boosting

In [ ]:
# max_iter=1000 is only a ceiling: early stopping picks the real number of trees
gb_base = make_pipe(HistGradientBoostingClassifier(
    max_iter=1000, early_stopping=True, random_state=RANDOM_STATE))

# Ranges instead of a fixed grid; loguniform suits the learning rate (0.02 and 0.2 differ by 10x)
gb_dist = {
    "model__learning_rate":    loguniform(0.02, 0.2),
    "model__max_leaf_nodes":   randint(15, 64),     # tree size = how many interactions each tree can capture
    "model__min_samples_leaf": randint(20, 101),    # bigger leaves = smoother, more regularized trees
}

gb_search = RandomizedSearchCV(
    gb_base, param_distributions=gb_dist,
    n_iter=10, scoring="roc_auc", cv=CV_SEARCH,
    n_jobs=1,                       # one fit at a time; the model already uses all cores itself
    random_state=RANDOM_STATE, return_train_score=True,
)
gb_search.fit(X_srch, y_srch)

print("best params:", gb_search.best_params_)
print("best CV ROC-AUC (30k sample):", round(gb_search.best_score_, 4))

# Top 5 settings with train vs CV score, to spot overfitting
gb_cols = ["param_model__learning_rate", "param_model__max_leaf_nodes",
           "param_model__min_samples_leaf", "mean_train_score", "mean_test_score"]
pd.DataFrame(gb_search.cv_results_)[gb_cols].sort_values("mean_test_score", ascending=False).head(5)

In [ ]:
# Re-score the winning settings with 5-fold CV on the FULL training set (same as every other row)
best_gb = {k.replace("model__", ""): v for k, v in gb_search.best_params_.items()}

gb_tuned = make_pipe(HistGradientBoostingClassifier(
    max_iter=1000, early_stopping=True, random_state=RANDOM_STATE, **best_gb))

results.append(cv_score("Gradient boosting (tuned)", gb_tuned, n_jobs=1))
print("trees used:", gb_tuned.fit(X_train, y_train).named_steps["model"].n_iter_)
pd.DataFrame(results)

### Insight: tuning Gradient Boosting

- **Best settings:** `learning_rate` ≈ 0.058, `max_leaf_nodes` = 50, `min_samples_leaf` = 43. Early stopping kept **310 trees**, which confirms that the 200-tree cap in Section 1 was cutting the model short. A lower learning rate takes smaller steps and needs more trees.
- **Small gain.** ROC-AUC 0.9099 → 0.9117 and F1 0.698 → 0.701. That's within fold-to-fold noise (`roc_auc_std`), so the defaults were already close to optimal. This is a normal, honest result, and it's why the plan says to cut tuning first when time is short.
- **Slight cost in overfitting.** The gap between train and CV F1 went from 0.036 to 0.047, still well below the forest's.
- **Small caveat:** the settings were chosen on a sample of the same training data they're scored on, so this row is slightly optimistic. The test set in Section 6 gives the unbiased number.

### 3.2 Random Forest

In [ ]:
# Random Forest search: how many features each split may see, leaf size, and a depth cap
rf_base = make_pipe(RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=RANDOM_STATE))

rf_dist = {
    "model__max_features":     ["sqrt", 0.1, 0.2, 0.3],   # sqrt = ~13 of 171 columns per split
    "model__min_samples_leaf": randint(1, 16),
    "model__max_depth":        [None, 20, 30],
}

rf_search = RandomizedSearchCV(
    rf_base, param_distributions=rf_dist,
    n_iter=8, scoring="roc_auc", cv=CV_SEARCH,
    n_jobs=1,                       # the forest already uses all cores itself (n_jobs=-1 inside)
    random_state=RANDOM_STATE, return_train_score=True,
)
rf_search.fit(X_srch, y_srch)

print("best params:", rf_search.best_params_)
print("best CV ROC-AUC (30k sample):", round(rf_search.best_score_, 4))

In [ ]:
# Re-score the winning forest with 5-fold CV on the full training set (200 trees, as in Part 1)
best_rf = {k.replace("model__", ""): v for k, v in rf_search.best_params_.items()}

rf_tuned = make_pipe(RandomForestClassifier(
    n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE, **best_rf))

results.append(cv_score("Random Forest (tuned)", rf_tuned, n_jobs=1))
pd.DataFrame(results).round(4)

### Insight: tuning Random Forest

- **Best settings:** `max_features` = 0.3 (about 51 of 171 columns per split, up from about 13 with the default `sqrt`), `min_samples_leaf` = 8, unlimited depth.
- **Tuning helped the forest much more than GB.** ROC-AUC 0.905 → 0.910 and F1 0.666 → 0.697, with recall rising from 0.59 to 0.65. With more features per split, each tree sees the strong signals (deposit type, lead time, parking) at most splits, so the trees get individually stronger. The trade-off is that they become more alike, so averaging removes less variance.
- **Now tied with GB.** ROC-AUC 0.9104 vs 0.9117, a gap of about 0.001, which is inside the fold-to-fold noise. The leaderboard alone can't pick a winner.
- **Tie-breakers:** the forest overfits more (train/CV F1 gap 0.089 vs 0.047). Sections 4 and 5 compare the curves and the probability quality.

## 4. ROC and Precision-Recall curves

Both curves sweep every possible threshold.
- **ROC:** true positive rate (recall) vs false positive rate. It measures **ranking quality**: how often a cancelling booking is scored above a staying one.
- **PR:** precision vs recall. It measures **how trustworthy the "cancel" flags are**. A random model scores a flat line at the cancel rate (0.275).

The probabilities are **out-of-fold**: every training booking is scored by a model that never saw it. These same probabilities are reused in Section 5 to choose the threshold, so the test set stays unused.

In [ ]:
# Out-of-fold probabilities for the two tuned finalists and the linear reference
models = {
    "GB (tuned)":    gb_tuned,
    "RF (tuned)":    rf_tuned,
    "LogReg (safe)": make_pipe(LogisticRegression(max_iter=1000)),
}

probas = {}
for name, pipe in models.items():
    probas[name] = cross_val_predict(
        pipe, X_train, y_train, cv=CV, method="predict_proba", n_jobs=1)[:, 1]

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 5))
for name, p in probas.items():
    RocCurveDisplay.from_predictions(y_train, p, name=name, ax=ax_roc)
    PrecisionRecallDisplay.from_predictions(y_train, p, name=name, ax=ax_pr)

ax_roc.plot([0, 1], [0, 1], "k--", label="random")
ax_roc.set_title("ROC curve")
base_rate = y_train.mean()                         # ~0.275: what a random model scores on PR
ax_pr.axhline(base_rate, color="k", ls="--", label=f"random ({base_rate:.3f})")
ax_pr.set_title("Precision-Recall curve")
ax_roc.legend(); ax_pr.legend()
plt.tight_layout(); plt.show()

# Numbers behind the curves (AP = average precision = area under the PR curve)
pd.DataFrame({
    name: {"ROC-AUC": roc_auc_score(y_train, p),
           "Avg precision": average_precision_score(y_train, p)}
    for name, p in probas.items()
}).T.round(4)

### Insight: ROC and PR curves

- **GB and RF overlap almost exactly** (ROC-AUC 0.91 each, AP 0.80 vs 0.79). GB is slightly higher at low recall, so its top-ranked bookings are a little purer. The tie holds.
- **PR shows the gap to Logistic Regression more clearly.** ROC-AUC differs by 0.06 (0.91 vs 0.85), but AP differs by 0.13 (0.80 vs 0.67). The false positive rate divides by all the bookings that *stay* (72.5% of the data), so even many false alarms barely move it. Precision divides by the bookings that were *flagged*, which is the group the hotel acts on. With imbalanced classes, PR is the more honest view of how many wasted actions you pay for.
- **The trade-off in business terms:** moving right along the PR curve means lowering the threshold. To catch about 80% of cancellations, the tree models give up roughly a third of their flags to false alarms (precision ≈ 0.65). Section 5 puts a price on that trade-off.
- **All models are far above random** (AP 0.80 vs 0.275), so the lift is about 2.9×.

## 5. Choosing the threshold by business cost

The default threshold of 0.5 assumes a missed cancellation and a false alarm cost the same. They don't:
- **Missed cancellation (FN):** the room stays empty, and that night's revenue is lost.
- **False alarm (FP):** a guest who would have stayed gets a reminder, a deposit request or an offer. That's a small cost and some annoyance.

`total_cost(t) = C_FN × FN(t) + C_FP × FP(t)`, with `C_FP = 1`. My estimate is that a miss costs **2×** a false alarm. Since that ratio is an assumption, 5× and 10× are checked as a sensitivity analysis.

**Theory check:** if the probabilities are well calibrated, the cheapest threshold is `C_FP / (C_FP + C_FN) = 1 / (1 + ratio)`, which gives 0.33, 0.17 and 0.09.

The sweep uses the **out-of-fold training probabilities** from Section 4. Tuning the threshold on the test set would leak the test set into a decision, and the final test score would be optimistic.

### 5.1 Cost sweep

In [ ]:
p_oof = probas["GB (tuned)"]             # out-of-fold P(cancel) for every training booking
y_np = y_train.to_numpy()                # NumPy array avoids index misalignment with p_oof

thresholds = np.arange(0.05, 0.96, 0.01)
ratios = [2, 5, 10]                      # C_FN / C_FP, with C_FP = 1

# Count FN and FP once per threshold; the cost for any ratio is then r * FN + FP
FN = np.array([((p_oof < t) & (y_np == 1)).sum() for t in thresholds])    # missed cancellations
FP = np.array([((p_oof >= t) & (y_np == 0)).sum() for t in thresholds])   # false alarms

fig, ax = plt.subplots(figsize=(8, 5))
best_t = {}
for r in ratios:
    cost = r * FN + FP
    i = cost.argmin()
    best_t[r] = round(float(thresholds[i]), 2)
    ax.plot(thresholds, cost, label=f"FN costs {r}x FP")
    ax.plot(thresholds[i], cost[i], "ko")                                 # mark the minimum
ax.set_xlabel("threshold"); ax.set_ylabel("total cost (in false-alarm units)")
ax.set_title("Cost vs threshold (out-of-fold GB probabilities)")
ax.legend(); plt.show()

# Best threshold per ratio vs the theory value 1 / (1 + ratio)
pd.DataFrame({
    "ratio (C_FN / C_FP)": ratios,
    "best threshold": [best_t[r] for r in ratios],
    "theory 1/(1+ratio)": [round(1 / (1 + r), 2) for r in ratios],
})

In [ ]:
# DECISION: a missed cancellation costs about 2x a false alarm -> use the cheapest threshold at ratio 2
CHOSEN_RATIO = 2
THRESHOLD = best_t[CHOSEN_RATIO]
print("chosen threshold:", THRESHOLD)

### Insight: cost sweep

| Ratio | Best threshold | Theory |
|---|---|---|
| 2× | 0.32 | 0.33 |
| 5× | 0.18 | 0.17 |
| 10× | 0.11 | 0.09 |

- **The empirical optimum matches the theory closely**, so GB's probabilities are well calibrated: a booking scored 0.30 really cancels about 30% of the time. If they hadn't matched, the probabilities would be over- or under-confident and would need calibration (Platt or isotonic) before being used for decisions.
- **The 2× curve is flat from about 0.25 to 0.45**, so the choice is robust. A small error in the threshold costs very little.
- **The cost ratio is the assumption that matters.** Going from 2× to 10× moves the threshold from 0.32 to 0.11, so all three are reported.
- **Chosen: 0.32 at 2×**, the conservative estimate. A flagged booking triggers a cheap action (a reminder or a soft deposit offer), so it costs much less than a lost room night, but not almost nothing.

### 5.2 Calibration check

The cost rule only works if the probabilities mean what they say. A calibration curve groups bookings by predicted probability and plots the actual cancel rate in each group. A perfect model lies on the diagonal. The **Brier score** is the mean squared error of the probabilities, where lower is better.

In [ ]:
# Calibration: predicted probability vs actual cancel rate, in 10 equal-sized groups (out-of-fold)
fig, ax = plt.subplots(figsize=(6, 5))
for name, p in probas.items():
    frac_pos, mean_pred = calibration_curve(y_np, p, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", label=f"{name} (Brier {brier_score_loss(y_np, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", label="perfect")
ax.set_xlabel("predicted probability"); ax.set_ylabel("actual cancel rate")
ax.set_title("Calibration (out-of-fold)"); ax.legend(); plt.show()

### Insight: calibration and the final model choice

- **GB and RF are both well calibrated.** Both sit almost on the diagonal, with Brier scores of 0.106 vs 0.107. I expected the forest's averaged votes to be worse calibrated, and the data shows they aren't, so calibration doesn't separate the two.
- **Logistic Regression is the miscalibrated one** (Brier 0.135). It under-predicts in the 0.3–0.45 range and over-predicts near 0.8.
- **Final model: Gradient Boosting (tuned).** It's tied with the forest on ROC-AUC, AP and calibration, and the remaining tie-breakers all favor GB:
  - **Overfitting:** the train/CV F1 gap is 0.047 vs 0.089.
  - **Model size:** about 300 shallow trees make a much smaller `.pkl` file than 200 deep trees, which matters for GitHub and Streamlit Cloud.
  - **SHAP:** explanations are faster to compute.

### 5.3 Operating point: the default 0.5 vs the cost-optimal threshold

In [ ]:
# For each cost ratio, compare the default 0.5 with that ratio's cheapest threshold (out-of-fold, training set)
rows = []
for r in ratios:
    for t in [0.5, best_t[r]]:
        pred = p_oof >= t
        fn = ((~pred) & (y_np == 1)).sum(); fp = (pred & (y_np == 0)).sum()
        rows.append({"ratio": f"{r}x", "threshold": t,
                     "precision": precision_score(y_np, pred), "recall": recall_score(y_np, pred),
                     "FN": fn, "FP": fp, "flagged share": pred.mean(), "cost": r * fn + fp})
op = pd.DataFrame(rows)
op["saving vs 0.5"] = 1 - op["cost"] / op.groupby("ratio")["cost"].transform("first")
op.round(3)

### Insight: operating point

| Ratio | Threshold | Precision | Recall | Missed (FN) | False alarms (FP) | Cost saving vs 0.5 |
|---|---|---|---|---|---|---|
| 2× | 0.50 → **0.32** | 0.740 → 0.645 | 0.666 → **0.822** | 6,331 → 3,373 | 4,452 → 8,597 | **−10%** |
| 5× | 0.50 → 0.18 | 0.740 → 0.546 | 0.666 → 0.931 | 6,331 → 1,314 | 4,452 → 14,693 | −41% |
| 10× | 0.50 → 0.11 | 0.740 → 0.486 | 0.666 → 0.968 | 6,331 → 598 | 4,452 → 19,416 | −63% |

- **At the chosen 0.32**, the model catches about 2,960 more cancellations at the cost of about 4,150 more false alarms, roughly 1.4 false alarms per extra catch. At 2× the break-even is 2 false alarms per catch, so lowering the threshold pays, and the saving is modest (10%) because it's close to break-even.
- **The more a miss costs, the more the default 0.5 hurts:** it wastes 10% of the cost at 2× and 63% at 10×. That's the main argument for choosing the threshold by cost and not accuracy.
- **There's a practical limit.** At 10× the model would flag about 55% of all bookings, and more than half of those flags would be false alarms. That's only acceptable if the action is very cheap (like a reminder email). A deposit request at that volume would annoy most guests.
- Costs aren't comparable across ratios because the units change. Compare each row only with its own 0.5 baseline.

## 6. Final test-set evaluation (run once)

The model (tuned GB) and the threshold (0.32) are now fixed. The model is refit on **all** training rows and scored **once** on the 17k held-out test bookings. After this, nothing is changed based on these numbers. Retuning to improve the test score would turn the test set into another validation set, and there would be no unbiased estimate left.

In [ ]:
# Refit the chosen model on ALL training data, then score the untouched test set at the chosen threshold
final_model = gb_tuned.fit(X_train, y_train)
p_test = final_model.predict_proba(X_test)[:, 1]
yt = y_test.to_numpy()

def summarize(y_true, prob, t, ratio=CHOSEN_RATIO):
    pred = prob >= t
    fn = ((~pred) & (y_true == 1)).sum(); fp = (pred & (y_true == 0)).sum()
    return {"ROC-AUC": roc_auc_score(y_true, prob),
            "Avg precision": average_precision_score(y_true, prob),
            "precision": precision_score(y_true, pred),
            "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred),
            f"cost per 1000 bookings ({ratio}x)": 1000 * (ratio * fn + fp) / len(y_true)}  # per 1000: sets differ in size

pd.DataFrame({"CV (train, out-of-fold)": summarize(y_np, p_oof, THRESHOLD),
              "Test (held out)": summarize(yt, p_test, THRESHOLD)}).T.round(3)

In [ ]:
# Test-set confusion matrix at the chosen threshold (rows = true class shares)
ConfusionMatrixDisplay.from_predictions(
    yt, p_test >= THRESHOLD, normalize="true", display_labels=["stays", "cancels"])
plt.title(f"Test set, threshold {THRESHOLD}"); plt.show()

### Insight: test-set result

| | CV (out-of-fold) | Test (held out) |
|---|---|---|
| ROC-AUC | 0.912 | **0.914** |
| Avg precision | 0.797 | 0.806 |
| Precision | 0.645 | 0.652 |
| Recall | 0.822 | **0.828** |
| F1 | 0.723 | 0.730 |
| Cost per 1000 bookings (2×) | 222.1 | 215.9 |

- **The model generalizes cleanly.** Every test metric is within about 0.01 of its CV estimate, so there are no signs of leakage or an optimistic CV.
- **The test set is slightly better, and that's not a red flag.** The final model trained on 100% of the training rows, while each CV fold trained on 80%. With about 4,700 test cancellations, recall has a standard error of about ±0.005, so the differences are around one standard error.
- **Headline:** on unseen bookings, the model catches **83% of cancellations** with a precision of 0.65 (ROC-AUC 0.914). 17% of guests who stay receive a false alarm. At a 2× cost ratio, this is about 10% cheaper than the default threshold.
- Majority baseline for comparison: ROC-AUC 0.50 and F1 0, so the success criterion (beat the baseline on F1 and ROC-AUC) is met by a wide margin.

## 7. SHAP explanations

The model says "0.82 chance of cancelling", and a hotel manager will ask why. SHAP splits each prediction into one contribution per feature:

`model output = base value + sum of feature contributions`

The units are **log-odds**. Positive values push toward "cancels" and negative values toward "stays". `TreeExplainer` computes exact SHAP values for tree models quickly. A sample of 2,000 test bookings is enough for stable plots.

In [ ]:
# Explain a 2000-booking sample of the test set with the final model
prep = final_model.named_steps["prep"]
gb = final_model.named_steps["model"]

X_shap_raw = X_test.sample(2000, random_state=RANDOM_STATE)
X_shap = prep.transform(X_shap_raw)
if hasattr(X_shap, "toarray"):
    X_shap = X_shap.toarray()                       # sparse -> dense for SHAP
feat_names = prep.get_feature_names_out()
X_shap = pd.DataFrame(np.asarray(X_shap), columns=feat_names)

explainer = shap.TreeExplainer(gb)
sv = explainer(X_shap)                              # one contribution per booking per feature

# Global view: which features matter, and in which direction (red = high value, blue = low value)
shap.plots.beeswarm(sv, max_display=15)

In [ ]:
# Local view: why was ONE booking flagged? Take the highest-risk booking in the sample
p_shap = final_model.predict_proba(X_shap_raw)[:, 1]
i = int(p_shap.argmax())
print("predicted cancel probability:", round(p_shap[i], 4))
shap.plots.waterfall(sv[i], max_display=12)

In [ ]:
# One importance per ORIGINAL feature: SHAP values add up, so sum the one-hot columns of each categorical column
sv_df = pd.DataFrame(sv.values, columns=feat_names)
cats_longest_first = sorted(cat_cols, key=len, reverse=True)   # so 'market_segment' isn't matched by a shorter name

groups = {}
for c in feat_names:
    base = c.split("__", 1)[1]                                 # strip the 'num__' / 'cat__' prefix
    if c.startswith("cat__"):
        base = next((cc for cc in cats_longest_first if base.startswith(f"{cc}_")), base)
    groups.setdefault(base, []).append(c)

grouped = pd.DataFrame({g: sv_df[cols].sum(axis=1) for g, cols in groups.items()})
imp = grouped.abs().mean().sort_values(ascending=False).head(15)

ax = imp.plot.barh(figsize=(7, 5)); ax.invert_yaxis()
plt.xlabel("mean |SHAP| (log-odds)"); plt.title("Feature importance by original feature")
plt.tight_layout(); plt.show()

### Insight: SHAP

**Global drivers (beeswarm).** These agree with the Logistic Regression coefficients from Part 1, so two very different models learned the same business logic:

| Feature | Effect | Business meaning |
|---|---|---|
| `required_car_parking_spaces` | High values push strongly toward "stays" (−5 to −8) | A guest who reserves parking is committed to coming. |
| `country_PRT` | Portuguese guests push toward "cancels" | Domestic bookings are cheaper to cancel and rebook. |
| `market_segment` (Online TA) | Pushes toward "cancels" | Online travel agency bookings are easy to cancel, and some guests book several options. |
| `total_of_special_requests` | Requests push toward "stays" | Guests who personalize a stay are engaged. |
| `lead_time` | Long lead times push toward "cancels" | More time for plans to change. Matches the LogReg weight (+0.48), and the wide spread shows the effect is non-linear. |
| `previous_cancellations` | Rare, but pushes very strongly toward "cancels" (up to +5) | Past behaviour predicts future behaviour. |
| `agent_9`, `agent_240` | Push toward "cancels" | Same agents LogReg flagged, probably group blocks that were cancelled in bulk. |

**One booking explained (waterfall).** The highest-risk booking has a model output of 8.78 log-odds, which is a cancel probability above 99.9%. Its story: a previous cancellation (+2.97), Portuguese (+1.98), a non-refundable deposit (+1.87 and +1.33), a long lead time (+1.05), and no parking or special requests. That's an explanation a hotel manager can act on.

**How to read the plots:**
- `deposit_type_Non Refund = 1` and `deposit_type_No Deposit = 0` carry the same information, so SHAP splits the credit between them. Read the group together, about +3.2 for deposit type. The grouped bar chart does this for every categorical column.
- The base value (−2.32 log-odds ≈ 9%) isn't the 27.5% cancel rate, because SHAP averages log-odds, and the average of log-odds isn't the log-odds of the average rate.

**Limitation: `arrival_date_year`.** 2015 bookings push toward "stays". The year doesn't describe a booking. It describes when the data was collected (for example, changes in cancellation policy or how bookings were recorded over 2015–2017). A random split mixes years, so the model is allowed to use this, but in deployment every new booking will be 2026 or later, outside the training range. Trees would treat them like the last year seen (2017), so this feature encodes a time trend that may no longer hold. Because the test set has already been used, the model isn't changed now. This is recorded as a limitation, and the fix for a production version is to drop the year and validate with a **time-based split** (train on earlier months, test on later ones).

## 8. Save the model and the final leaderboard

In [ ]:
# Save the pipeline AND the threshold together: the app needs both to turn a probability into a decision
Path("../models").mkdir(exist_ok=True)
artifact = {
    "pipeline": final_model,                       # preprocessing + tuned GB, fitted on all training data
    "threshold": THRESHOLD,                        # chosen at a 2x cost ratio (Section 5)
    "cost_ratio": CHOSEN_RATIO,
    "features": X_train.columns.tolist(),          # column order the app must supply
    "leaky_dropped": LEAKY,
    "sklearn_version": sklearn.__version__,        # a .pkl only loads reliably on the same version
}
joblib.dump(artifact, "../models/cancel_model.pkl")

# Reload test: the saved model must give identical predictions
loaded = joblib.load("../models/cancel_model.pkl")
assert np.allclose(loaded["pipeline"].predict_proba(X_test)[:, 1], p_test)
print("saved; size (MB):", round(Path("../models/cancel_model.pkl").stat().st_size / 1e6, 2),
      "| sklearn", sklearn.__version__)

In [ ]:
# Final leaderboard: Part 1 + Part 2 rows, 5-fold CV on the training set, sorted by ROC-AUC
day4 = pd.DataFrame(results)
final_lb = (pd.concat([day3, day4], ignore_index=True)
              .drop_duplicates(subset="model", keep="last")          # baseline appears in both parts
              .assign(gap_f1=lambda d: d["train_f1"] - d["f1"])
              .sort_values("roc_auc", ascending=False)
              .reset_index(drop=True))

Path("../reports").mkdir(exist_ok=True)
day4.to_csv("../reports/day4_leaderboard.csv", index=False)
final_lb.to_csv("../reports/final_classification_leaderboard.csv", index=False)
final_lb[["model", "roc_auc", "f1", "precision", "recall", "gap_f1"]].round(3)

### Insight: saving the model

- **The pipeline and the threshold are saved together.** The app needs both to turn a probability into a decision. If the app used the default 0.5, it would silently undo the cost-based choice from Section 5.
- **The reload test passed.** The saved file gives exactly the same test predictions as the model in memory, so the Streamlit app will behave like this notebook.
- **The scikit-learn version is recorded**, because a `.pkl` file only loads reliably on the same version. Pin it in `requirements.txt` before deploying.
- **The final leaderboard combines Part 1 and Part 2**, with 11 models from the majority baseline (0.50 ROC-AUC) to tuned Gradient Boosting (0.912). It's saved to `reports/final_classification_leaderboard.csv` for the README.

## 9. Summary: models and decisions

*Numbers are from the run on 25 Nov 2026. A rerun may change the third decimal.*

**Final leaderboard** (5-fold CV on the training set, threshold 0.5, sorted by ROC-AUC; Part 1 rows included):

| Model | ROC-AUC | F1 | Precision | Recall | Train/CV F1 gap |
|---|---|---|---|---|---|
| **Gradient Boosting (tuned)** | **0.912** | **0.701** | 0.740 | 0.666 | 0.047 |
| Random Forest (tuned) | 0.910 | 0.697 | 0.749 | 0.651 | 0.089 |
| Gradient Boosting (default) | 0.910 | 0.698 | 0.739 | 0.661 | 0.036 |
| Random Forest (Part 1) | 0.905 | 0.666 | 0.770 | 0.588 | 0.072 |
| MLP (64, 32) | 0.901 | 0.685 | 0.720 | 0.654 | 0.050 |
| Decision Tree (depth 12) | 0.881 | 0.671 | 0.699 | 0.645 | 0.041 |
| SVM RBF (20k sample) | 0.868 | 0.625 | 0.742 | 0.540 | 0.038 |
| Logistic Regression (safe) | 0.855 | 0.578 | 0.671 | 0.507 | 0.002 |
| KNN (k=51) | 0.835 | 0.559 | 0.696 | 0.467 | 0.025 |
| Naive Bayes | 0.749 | 0.538 | 0.400 | 0.823 | 0.002 |
| Majority baseline | 0.500 | 0.000 | 0.000 | 0.000 | 0.000 |

**Final model:** tuned Gradient Boosting at **threshold 0.32**, saved to `models/cancel_model.pkl`.

**Test-set result (held out, used once):** ROC-AUC 0.914, recall 0.83, precision 0.65, F1 0.73. It's within about 0.01 of CV, and both beat the majority baseline (0.50 / 0) by a wide margin.

### Key findings
- **Tree ensembles win.** Boosting, forest and MLP all reach 0.90–0.91 ROC-AUC. Lead time, deposit type and market segment have non-linear, interacting effects, which trees learn directly.
- **Boosting and bagging tie after tuning.** Tuning mattered more for the forest (+0.005 AUC, +0.03 F1, via `max_features`) than for GB (+0.002), whose defaults were already near-optimal.
- **The threshold matters more than tuning.** Moving from 0.5 to 0.32 raises recall from 0.67 to 0.82 and cuts the cost by 10% at 2×, and by up to 63% at 10×. Tuning added about 0.002 AUC.
- **The probabilities are trustworthy.** The cost-optimal thresholds match `1/(1+ratio)`, and the calibration curves sit on the diagonal (Brier 0.106). So the risk score can be shown to staff as a real probability.
- **The model is explainable.** SHAP agrees with the Logistic Regression coefficients: parking, special requests and deposit type separate committed guests from likely cancellations.

### Decision log
| Decision | Reason |
|---|---|
| Dropped `assigned_room_type`, `booking_changes` | Only known after booking (leakage). It cost about 0.008 AUC, and the honest number is the one reported. |
| Compared models on ROC-AUC, not accuracy or F1 | ROC-AUC doesn't depend on the threshold. The threshold is chosen separately, by cost. |
| Light tuning (30k sample, 3 folds, 8–10 settings) | The full search overheated the laptop. The search only ranks settings, and the winners were re-scored on the full data. |
| Tuned the top two models (GB and RF) | To compare them fairly. A tuned model against an untuned one isn't an even contest. |
| Chose GB over RF | Tied on AUC, AP and calibration. GB overfits less (0.047 vs 0.089), is smaller to deploy, and is faster to explain. |
| Threshold 0.32 (2× cost ratio) | A missed cancellation loses a room night. A false alarm triggers a cheap action. 5× and 10× are reported as sensitivity. |
| Chose the threshold on out-of-fold training predictions | Using the test set for any decision would make the final score optimistic. |
| Used the test set once, with no changes afterwards | To keep one unbiased estimate. |
| Kept `arrival_date_year`, flagged it as a limitation | It's a time proxy that won't hold in the future. Fixing it after seeing test results would contaminate the test set. Next version: drop it and use a time-based split. |

### Model cheat sheet rows (new in Part 2)
| Model | How it works (1 line) | Key hyperparameters | Fails when | Needs scaling? |
|---|---|---|---|---|
| Gradient Boosting | Shallow trees added one at a time, each fitting the current errors | `learning_rate`, `max_iter` (+ early stopping), `max_leaf_nodes`, `min_samples_leaf` | Learning rate too high or too many trees without early stopping (overfits); noisy labels | No |
| MLP | Layers of weighted sums + ReLU, trained by gradient descent | `hidden_layer_sizes`, `alpha`, `learning_rate_init`, `early_stopping` | Small or tabular data with threshold-like effects; unscaled inputs | Yes |

### Next steps
- **Day 5:** Module C (unsupervised guest segments).
- **Day 6:** load `cancel_model.pkl` in the Streamlit "Cancellation Risk" tab. Show the probability, the flag at 0.32, a suggested action, and the SHAP waterfall for the booking.
- **Later:** a time-based validation split without `arrival_date_year`.